# 01 – Análise Exploratória de Dados (EDA) – MetroPT-3

**Dataset:** MetroPT-3 Air Compressor (UCI ML Repository, 2022)  
**Objetivo:** Compreender a distribuição dos sinais de sensores, identificar padrões de falha e construir intuição para a etapa de feature engineering.

> **Pré-requisito:** Execute o script de ingestão antes de abrir este notebook:  
> ```bash
> python src/ingest_metropt.py
> ```

## 0. Configuração do Ambiente

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Inline plots
%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (14, 5)})

# Paths
NOTEBOOK_DIR: Path = Path().resolve()
PARQUET_PATH: Path = NOTEBOOK_DIR.parent / "data" / "processed" / "metropt3.parquet"

print(f"Parquet path: {PARQUET_PATH}")
print(f"Exists: {PARQUET_PATH.exists()}")

## 1. Carregamento dos Dados

In [ ]:
df: pd.DataFrame = pd.read_parquet(PARQUET_PATH)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nPrimeiras linhas:")
df.head()

## 2. Estatísticas Descritivas

In [ ]:
df.describe(include="all").T

In [ ]:
# Verificar valores nulos por coluna
null_counts: pd.Series = df.isnull().sum()
print("Valores nulos por coluna:")
print(null_counts[null_counts > 0] if null_counts.any() else "Nenhum valor nulo encontrado.")

## 3. Distribuição de Classes (Label)

> **Nota:** O MetroPT-3 é um dataset de anomalia semi-supervisionado. Os rótulos de falha  
> (quando disponíveis) indicam eventos de falha específicos. Ajuste o nome da coluna de label  
> conforme a versão do dataset utilizada.

In [ ]:
# TODO: Ajuste 'label_col' para o nome correto da coluna de rótulo do dataset.
# Exemplo: algumas versões usam 'failure', outras 'label' ou 'target'.

LABEL_COL: str = "label"  # <- altere se necessário

if LABEL_COL in df.columns:
    class_counts: pd.Series = df[LABEL_COL].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    class_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "tomato"])
    axes[0].set_title("Contagem por Classe")
    axes[0].set_xlabel("Classe")
    axes[0].set_ylabel("Quantidade")

    class_counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Proporção de Classes")
    axes[1].set_ylabel("")

    plt.suptitle("Distribuição de Classes – MetroPT-3", fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f"\nDesbalanceamento (razão maioria/minoria): {class_counts.max() / class_counts.min():.1f}x")
else:
    print(f"Coluna '{LABEL_COL}' não encontrada. Colunas disponíveis: {list(df.columns)}")

## 4. Séries Temporais dos Sensores

In [ ]:
SENSOR_COLS: list[str] = ["TP2", "TP3", "H1", "DV_pressure", "Motor_current", "Oil_temperature"]
SAMPLE_HOURS: int = 24  # Plota as primeiras N horas para não sobrecarregar o gráfico

if "timestamp" in df.columns:
    df_plot: pd.DataFrame = df.set_index("timestamp").sort_index()
    cutoff = df_plot.index[0] + pd.Timedelta(hours=SAMPLE_HOURS)
    df_plot = df_plot.loc[:cutoff]
else:
    df_plot = df.copy()

available_sensors: list[str] = [c for c in SENSOR_COLS if c in df_plot.columns]

fig, axes = plt.subplots(len(available_sensors), 1, figsize=(14, 3 * len(available_sensors)), sharex=True)
if len(available_sensors) == 1:
    axes = [axes]

for ax, col in zip(axes, available_sensors):
    ax.plot(df_plot.index, df_plot[col], linewidth=0.6, color="steelblue")
    ax.set_ylabel(col, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_title(f"Séries Temporais dos Sensores (primeiras {SAMPLE_HOURS}h)", fontsize=13)
axes[-1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

## 5. Distribuição Individual dos Sensores (Histogramas)

In [ ]:
numeric_cols: list[str] = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols: int = 3
n_rows: int = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes_flat = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes_flat[i].hist(df[col].dropna(), bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
    axes_flat[i].set_title(col, fontsize=10)
    axes_flat[i].set_ylabel("Frequência")
    axes_flat[i].grid(True, alpha=0.3)

# Oculta subplots extras
for j in range(len(numeric_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Distribuição dos Sensores – MetroPT-3", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Matriz de Correlação

In [ ]:
corr_matrix: pd.DataFrame = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)

# Anotações numéricas
for row_idx in range(len(corr_matrix)):
    for col_idx in range(len(corr_matrix.columns)):
        val: float = corr_matrix.iloc[row_idx, col_idx]
        ax.text(col_idx, row_idx, f"{val:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(val) > 0.6 else "black")

ax.set_title("Matriz de Correlação – Sensores MetroPT-3", fontsize=14)
plt.tight_layout()
plt.show()

# Top correlações (excluindo diagonal)
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print("\nTop 10 pares mais correlacionados:")
print(corr_pairs.head(10).to_string())

## 7. Detecção de Outliers (IQR)

Identifica amostras fora do intervalo interquartil para os sensores principais.

In [ ]:
KEY_SENSORS: list[str] = [c for c in ["TP2", "TP3", "H1", "DV_pressure", "Motor_current"] if c in df.columns]

outlier_summary: list[dict] = []
for col in KEY_SENSORS:
    q1: float = df[col].quantile(0.25)
    q3: float = df[col].quantile(0.75)
    iqr: float = q3 - q1
    lower: float = q1 - 1.5 * iqr
    upper: float = q3 + 1.5 * iqr
    n_outliers: int = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_summary.append({"sensor": col, "q1": q1, "q3": q3, "iqr": iqr,
                             "lower_fence": lower, "upper_fence": upper,
                             "n_outliers": n_outliers, "pct": n_outliers / len(df) * 100})

pd.DataFrame(outlier_summary).set_index("sensor").round(4)

## 8. Próximos Passos

- [ ] Feature Engineering: janelamento temporal (rolling mean/std), lag features  
- [ ] Normalização/padronização dos sensores  
- [ ] Treino do modelo baseline (IsolationForest / LSTM-AE)  
- [ ] Exportação do modelo treinado para formato ONNX (requisito de produção)